# Note

This code is based on the one developed by Gilad Wisney in:

https://towardsdatascience.com/deep-reinforcement-learning-and-monte-carlo-tree-search-with-connect-4-ba22a4713e7a

https://github.com/giladariel/Connect4

It also uses some code from Miguel Teixara's work.

https://repositorio-aberto.up.pt/handle/10216/135628

https://github.com/ajmcastro/quantum-reinforcement-learning

Check README.txt for more details.

# Reload

In [1]:
%load_ext autoreload
%autoreload 2

# Imports

In [2]:
from connect4 import ConnectFour
from connect4_agent import ConnectFourAgent
from connect4_agent_greedy import ConnectFourAgentGreedy
from connect4_agent_with_tags import ConnectFourAgentWithTags
from randomized_negamax import RandomizedNegamax

import random
import numpy as np
from prettytable import PrettyTable
import datetime
import time
%matplotlib qt 
import matplotlib.pyplot as plt

# Seeds

In [3]:
seeds1 = [83492, 97812, 21008, 50978, 89445, 15226, 31115, 82311, 50023, 61977,
          10056, 71165, 60570, 10935, 51107, 46678, 70653, 98105, 38160, 61106] #Classical_Epsilon_Greedy_vs_RN2
seeds2 = [84398, 83286, 48132, 93932, 32489, 94829, 43829, 83274, 63781, 48821,
          23894, 39265, 92852, 10099, 89324, 22044, 79152, 47225, 14397, 98327] #Classical_Tags_vs_RN2
seeds3 = [74929, 29424, 93829, 63920, 27482, 77402, 83891, 38912, 84329, 92382,
          73875, 62428, 54326, 10382, 94213, 88269, 29237, 82013, 49285, 66637] #Quantum_Tags_vs_RN2
seeds4 = [73522, 59021, 93274, 48293, 19083, 32879, 58329, 32895, 74365, 62762,
          87924, 32784, 78329, 65681, 45324, 44007, 15390, 81015, 61577, 52107] #RN2_vs_Classical_Tags
seeds5 = [84012, 72958, 42442, 25163, 33398, 15296, 75499, 12374, 26651, 50729, 
          83924, 78324, 93823, 52476, 74630, 86161, 60015, 71555, 56701, 64302] #RN2_vs_Quantum_Tags

# Experiments

In [4]:
experiment1 = "Experiment1_C_Epsilon_Greedy_vs_RN2" #Classical Epsilon-Greedy vs Randomized Negamax (lookahead = 2)
experiment2 = "Experiment2_C_Tags_vs_RN2" #Classical Tags vs Randomized Negamax (lookahead = 2)
experiment3 = "Experiment3_Q_Tags_vs_RN2" #Quantum Tags vs Randomized Negamax (lookahead = 2)
experiment4 = "Experiment4_RN2_vs_C_Tags" #Randomized Negamax (lookahead = 2) vs Classical Tags
experiment5 = "Experiment5_RN2_vs_Q_Tags" #Randomized Negamax (lookahead = 2) vs Quantum Tags

seeds_dict = {experiment1:seeds1, experiment2:seeds2, experiment3:seeds3, experiment4:seeds4, experiment5:seeds5}
episodes_train_dict = {experiment1:1800, experiment2:1800, experiment3:1800, experiment4:3600, experiment5:3600}
player_dict = {experiment1:1, experiment2:1, experiment3:1, experiment4:2, experiment5:2}

# Parameters

In [5]:
player1_num = 1
player1_tag = 1
player2_num = 2
player2_tag = -1

max_memory = 300 #Max memory size for the learning agents

starting_epsilon = 1 #Starting Epsilon for the Epsilon-Greedy agents
constant_epsilon = False #For the Epsilon-Greedy agents

r = 5 #Max number of reflections for the agents using tags
player1_delta = 97 #Player1 delta for the agents using tags
player2_delta = 194 #Player2 delta for the agents using tags

#Boolean for the isQuantum parameter
not_quantum = False #Classical agent
is_quantum = True #Quantum agent

lookahead = 2 #Lookahead for the Randomized Negamax Agent

episodes_test = 1000 #Number of episodes used for testing each agent

# Player 1 and Player 2 Agents

In [6]:
def getPlayer1(experiment, seed):
    if (experiment == experiment1):
        agent1 = ConnectFourAgentGreedy(player1_num, max_memory, seed, experiment, starting_epsilon, constant_epsilon)
    elif (experiment == experiment2):
        agent1 = ConnectFourAgentWithTags(player1_num, max_memory, seed, experiment, not_quantum, r, player1_delta)
    elif (experiment == experiment3):
        agent1 = ConnectFourAgentWithTags(player1_num, max_memory, seed, experiment, is_quantum, r, player1_delta)
    elif ((experiment == experiment4) or (experiment == experiment5)):
        agent1 = RandomizedNegamax(player1_tag, lookahead)
        
    return agent1

def getPlayer2(experiment, seed):
    if ((experiment == experiment1) or (experiment == experiment2) or (experiment == experiment3)):
        agent2 = RandomizedNegamax(player2_tag, lookahead)
    elif (experiment == experiment4):
        agent2 = ConnectFourAgentWithTags(player2_num, max_memory, seed, experiment, not_quantum, r, player2_delta)
    elif (experiment == experiment5):
        agent2 = ConnectFourAgentWithTags(player2_num, max_memory, seed, experiment, is_quantum, r, player2_delta)
        
    return agent2

# Training and Testing the Agents

In [7]:
for experiment in seeds_dict:        
    episodes_train = episodes_train_dict[experiment]

    seeds = seeds_dict[experiment]
    for seed in seeds:
        random.seed(seed)

        player1 = getPlayer1(experiment, str(seed)) #Player 1 Agent
        player2 = getPlayer2(experiment, str(seed)) #Player 2 Agent
        
        #Only Train and Test if the experiment hasn't already finished for that seed
        if ( (not player1.is_experiment_done()) and (not player2.is_experiment_done()) ):
            print("-------------------Training - Seed", seed, '-------------------\n')
            game = ConnectFour(player1, player2, episodes_train, learn=True) #Training
            game.play_multiple_games() #Train the Agent

            print("-------------------Testing - Seed", seed, '-------------------\n')
            game = ConnectFour(player1, player2, episodes_test, learn=False) #Testing
            game.play_multiple_games() #Test the Agent

# Calculate Statistics

In [8]:
plus_minus = " \u00B1 "

def player_to_player_string(player_num):
    if player_num == 1:
        return 'First'
    elif player_num == 2:
        return 'Second'

def get_table_row(mean, std, number_of_decimals): #Round but keep trailing zeroes
    table_row = '{0:.{1}f}'.format(mean, number_of_decimals) + plus_minus + '{0:.{1}f}'.format(std, number_of_decimals)
    return table_row

def statistic_to_table_row(statistic, number_of_decimals):
    mean = np.mean(statistic)
    std =  np.std(statistic)
    return get_table_row(mean, std, number_of_decimals)

def relative_statistic_to_table_row(statistic, number_of_decimals, episodes): #Represent from 0-100%
    mean = np.mean(statistic) / episodes * 100
    std =  np.std(statistic) / episodes * 100
    return get_table_row(mean, std, number_of_decimals)

# (example) statistics_for_each_seed -> [ [ep1_reward, ep2_reward, ...], [ep1_reward, ep2_reward, ...], ... ]
#                                                  ^Seed1                          ^Seed2
# Note: This example uses 'reward' as the statistic, but it applies to the other statistics as well
def get_mean_per_episode(episodes_train, statistics_for_each_seed):
    episodes_mean_statistic_list = [] #Mean statistic for each episode -> [mean_statistic_ep1, mean_statistic_ep2, ...]
    for episode in range(episodes_train):
        episode_statistics = [] #Every seed's statistic for this episode
        for seed_statistics in statistics_for_each_seed: #Iterate the statistics for each seed
            episode_statistics.append(seed_statistics[episode]) #Append that seed's statistic for this episode
        episodes_mean_statistic_list.append( np.mean(episode_statistics) ) #Mean statistic for this episode
        
    return episodes_mean_statistic_list

def get_moving_average(data, window):
    cum_sum_vec = np.cumsum(np.insert(data, 0, 0)) 
    moving_average_vec = ( cum_sum_vec[window:] - cum_sum_vec[:-window] ) / window
    return moving_average_vec

In [9]:
training_table = PrettyTable(['Agent', 'Moves', 'States', 'Iterations'])
extra_training_table = PrettyTable(['Agent', 'Time', 'Wins', 'Losses', 'Draws'])
testing_table = PrettyTable(['Agent', 'Wins', 'Moves'])
extra_testing_table = PrettyTable(['Agent', 'Losses', 'Draws'])

fig, axs = plt.subplots(2, 2, figsize=(17, 17), sharex = True)
font_size = 18

for experiment in seeds_dict:
    seeds = seeds_dict[experiment]
    episodes_train = episodes_train_dict[experiment]
    player_num = player_dict[experiment]
    player_string = player_to_player_string(player_num)
    
    training_moves, training_states_explored, training_iterations = [], [], []
    training_time, training_wins, training_losses, training_draws = [], [], [], []
    testing_wins, testing_losses, testing_draws, testing_moves = [], [], [], []
    
    all_seeds_table = PrettyTable(['Seed', 'Training Moves', 'States', 'Iterations', 'Testing Wins', 'Testing Moves'])
    
    rewards_for_each_seed, wins_for_each_seed, moves_for_each_seed, states_for_each_seed = [], [], [], []
    
    for seed in seeds:
        agent = ConnectFourAgent(player_num, max_memory, str(seed), experiment)
        statistics = agent.load_statistics()
        
        #Training Statistics
        training_moves.append(statistics["Training - Number of Moves"])
        training_states_explored.append(statistics["Training - States Explored"])
        if ("Training - Mean_Iterations_Flagged_Action" in statistics): #Only collected for agents using flags
            training_iterations.append(statistics["Training - Mean_Iterations_Flagged_Action"])
            iterations_row = np.round(np.array(statistics["Training - Mean_Iterations_Flagged_Action"]), 3)
        else:
            iterations_row = ''
        
        t = time.strptime(statistics["Training - Time"], '%H:%M:%S.%f')
        time_in_seconds = datetime.timedelta(hours=t.tm_hour, minutes=t.tm_min, seconds=t.tm_sec).total_seconds()
        training_time.append(time_in_seconds)
        training_wins.append(statistics["Training - Wins"])
        training_losses.append(statistics["Training - Losses"])
        training_draws.append(statistics["Training - Draws"])
        
        #Testing Statistics
        testing_wins.append(statistics["Testing - Wins"])
        testing_losses.append(statistics["Testing - Losses"])
        testing_draws.append(statistics["Testing - Draws"])
        testing_moves.append(statistics["Testing - Number of Moves"])
    
        #Statistics for the all_seeds_table
        all_seeds_table.add_row([seed,
                                 np.round(statistics["Training - Number of Moves"], 3),
                                 np.round(statistics["Training - States Explored"], 0),
                                 iterations_row,
                                 np.round(statistics["Testing - Wins"] / episodes_test * 100, 1),
                                 np.round(statistics["Testing - Number of Moves"], 3)])
        
        #Statistics for the plots
        rewards_for_each_seed.append(statistics["Training_Rewards_List"]) #Append this seed's rewards list
        wins_for_each_seed.append(statistics["Training_Wins_List"]) #Append this seed's wins list
        moves_for_each_seed.append(statistics["Training_Moves_List"]) #Append this seed's moves list
        states_for_each_seed.append(statistics["Training_States_List"]) #Append this seed's states list
        
    #############Plots for the evolution of statistics over the episodes#############
    window = 50 #Window used for the moving average
    episodes_mean_reward_list = get_moving_average( get_mean_per_episode(episodes_train, rewards_for_each_seed), window )
    episodes_mean_wins_list = get_moving_average( get_mean_per_episode(episodes_train, wins_for_each_seed), window )
    episodes_mean_moves_list = get_moving_average( get_mean_per_episode(episodes_train, moves_for_each_seed), window )
    episodes_mean_states_list = get_moving_average( get_mean_per_episode(episodes_train, states_for_each_seed), window )
    
    axs[0, 0].plot(np.arange(window, episodes_train+1), episodes_mean_reward_list, label=experiment)
    axs[0, 0].set_title('Reward per Episode', fontweight='bold', size=font_size)
    axs[0, 0].set_ylabel('Reward', fontsize = font_size)
    
    #Convert wins to relative values (0-100%) instead of absolute values (0-1)
    axs[0, 1].plot(np.arange(window, episodes_train+1), np.array(episodes_mean_wins_list)*100, label=experiment)
    axs[0, 1].set_title('Win% per Episode', fontweight='bold', size=font_size)
    axs[0, 1].set_ylabel('Win%', fontsize = font_size)
    
    axs[1, 0].plot(np.arange(window, episodes_train+1), episodes_mean_moves_list, label=experiment)
    axs[1, 0].set_title('Number of Moves per Episode', fontweight='bold', size=font_size)
    axs[1, 0].set_ylabel('Number of Moves', fontsize = font_size)
    axs[1, 0].set_xlabel('Episode', fontsize = font_size)
    
    axs[1, 1].plot(np.arange(window, episodes_train+1), episodes_mean_states_list, label=experiment)
    axs[1, 1].set_title('Number of States Explored per Episode', fontweight='bold', size=font_size)
    axs[1, 1].set_ylabel('Number of States', fontsize = font_size)
    axs[1, 1].set_xlabel('Episode', fontsize = font_size)
        
    #############Statistics(all seeds for this experiment)#############
    print("\n\nStatistics for all seeds of " + experiment)
    print(all_seeds_table)
    
    
    #############Statistics(mean for each experiment)#############
    if (not training_iterations): #Check if it's empty
        training_iterations_row = ''
    else:
        training_iterations_row = statistic_to_table_row(training_iterations, 3)
        
    training_table.add_row([experiment,
                            statistic_to_table_row(training_moves, 3),
                            statistic_to_table_row(training_states_explored, 0),
                            training_iterations_row])
    
    mean_time = datetime.timedelta( seconds = round(np.mean(training_time)) )
    std_time = datetime.timedelta( seconds = round(np.std(training_time)) )
    
    extra_training_table.add_row([experiment,
                                  str(mean_time) + plus_minus + str(std_time),
                                  relative_statistic_to_table_row(training_wins, 1, episodes_train),
                                  relative_statistic_to_table_row(training_losses, 1, episodes_train),
                                  relative_statistic_to_table_row(training_draws, 1, episodes_train)])
    
    testing_table.add_row([experiment,
                           relative_statistic_to_table_row(testing_wins, 1, episodes_test),
                           statistic_to_table_row(testing_moves, 3)])
    
    extra_testing_table.add_row([experiment,
                                 relative_statistic_to_table_row(testing_losses, 1, episodes_test),
                                 relative_statistic_to_table_row(testing_draws, 1, episodes_test)])
    
#Plots
for ax in fig.get_axes():
    ax.legend()
plt.show()    

print("\n\nTraining Statistics")
print(training_table)
print(extra_training_table)
print("\n\nTesting Statistics")
print(testing_table)
print(extra_testing_table)



Statistics for all seeds of Experiment1_C_Epsilon_Greedy_vs_RN2
+-------+----------------+--------+------------+--------------+---------------+
|  Seed | Training Moves | States | Iterations | Testing Wins | Testing Moves |
+-------+----------------+--------+------------+--------------+---------------+
| 83492 |     22.283     | 11270  |            |     16.4     |     28.932    |
| 97812 |     23.35      | 12187  |            |     25.4     |     29.226    |
| 21008 |     20.764     |  9305  |            |     17.6     |     30.456    |
| 50978 |     23.048     | 11485  |            |     71.7     |     14.895    |
| 89445 |     18.28      |  7933  |            |     74.2     |      11.8     |
| 15226 |     16.688     |  6426  |            |     72.7     |     12.093    |
| 31115 |     15.757     |  6333  |            |     82.0     |     11.236    |
| 82311 |     22.991     | 11490  |            |     61.6     |     19.464    |
| 50023 |     24.229     | 13113  |            |     3